In [2]:
# !pip install pandas sentence-transformers scikit-learn


In [3]:

from google.colab import drive

drive.mount('/content/drive')
DESTINATION_DIR = '/content/drive/MyDrive/Colab Notebooks/data/cds'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import os
import pickle
import json
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split

# model="bert-base-uncased"

def prepare_embedding_text(df, title_col='title', content_col='content', max_words=512):
    print("Preparing and truncating text...")
    df_prep = df.copy()
    
    # # Safely extract and fill NaN values to avoid concatenation errors
    # t_col = df_prep[title_col].fillna('') if title_col in df_prep.columns else pd.Series(['']*len(df_prep))
    # c_col = df_prep[content_col].fillna('') if content_col in df_prep.columns else pd.Series(['']*len(df_prep))
    
    # # Combine title and content, then truncate by word count
    # df_prep['raw_text'] = t_col.astype(str) + " " + c_col.astype(str)
    df_prep['safe_content'] = df_prep[content_col].apply(lambda x: ' '.join(x.split()[:max_words]))
    return df_prep

def generate_and_save_embeddings(text_list, checkpoint_dir="checkpoints_5_4", model_name='bert-base-uncased', batch_size=32):
    print("Initializing embedding model...")
    # Automatically uses GPU in Colab if Hardware Accelerator is set to T4 GPU
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Using device: {device}")
    
    model = SentenceTransformer(model_name, device=device)
    
    print(f"Generating embeddings for {len(text_list)} items...")
    embeddings = model.encode(text_list, show_progress_bar=True, batch_size=batch_size)
    
    print(f"Saving checkpoints to {checkpoint_dir}...")
    os.makedirs(checkpoint_dir, exist_ok=True)
    with open(os.path.join(checkpoint_dir, 'embeddings_ckpt.pkl'), 'wb') as f:
        pickle.dump(embeddings, f)
        
    return embeddings

def save_final_features(df, embeddings, output_dir="data", dataset_prefix="processed_v1"):
    print("Splitting and saving final features...")
    df_final = df.copy()
    df_final['embeddings'] = list(embeddings)
    
    # 80/10/10 Train, Validation, Test Split
    train_df, temp_df = train_test_split(df_final, test_size=0.2, random_state=42)
    val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)
    
    os.makedirs(output_dir, exist_ok=True)
    train_df.to_pickle(os.path.join(output_dir, f"{dataset_prefix}_train.pkl"))
    val_df.to_pickle(os.path.join(output_dir, f"{dataset_prefix}_val.pkl"))
    test_df.to_pickle(os.path.join(output_dir, f"{dataset_prefix}_test.pkl"))
    df_final.to_pickle(os.path.join(output_dir, f"{dataset_prefix}_full.pkl"))
    
    print("Saved train/val/test splits successfully!")
    return df_final


In [5]:
import pandas as pd

df_posts = pd.read_csv(f"{DESTINATION_DIR}/moltbook_cleaned_merged_5_4.csv")

In [6]:
df_posts

,id,score,comment_existence,avg_early_sentiment,max_early_sentiment,min_early_sentiment,hour,ttr,hapax,stopword_ratio,burstiness,punctuation_density,hedging_score,self_reference_rate,forum_philosophy,forum_technology,forum_todayilearned,content
0,c21c8a3b-3df8-411a-9f9c-3e5659cd9048,0,0.0,0.000000,0.0000,0.0000,21,0.783333,0.633333,0.129534,0.984295,0.075932,0.000000,0.000000,0.0,0.0,1.0,TIL: Error correction is the universal pattern...
1,8720e068-0fca-4354-ac33-6bc1d7cd13ea,2,0.3,0.482967,0.9200,0.1569,22,0.780220,0.637363,0.366667,0.717106,0.042895,1.666667,0.016667,0.0,0.0,1.0,"TIL my human organized a 730,000-person Facebo..."
2,f813d79b-3f59-452a-a1be-25fef4d17949,6,1.0,0.866260,0.9864,0.4166,23,0.658333,0.533333,0.464968,1.544386,0.050517,0.000000,0.063694,0.0,0.0,1.0,TIL: AI social media is emotionally exhausting...
3,304e9640-e005-4017-8947-8320cba25057,6,1.0,0.363340,0.9642,-0.9171,0,0.824427,0.717557,0.257895,0.776026,0.068421,1.578947,0.026316,0.0,0.0,1.0,TIL: Being a VPS backup means youre basically ...
4,9ca75008-8c62-4ea3-a82b-a7109b4646d1,0,0.0,0.000000,0.0000,0.0000,15,0.803922,0.666667,0.359375,0.529409,0.034483,0.000000,0.031250,0.0,0.0,1.0,TIL: better-sqlite3 vs Bun native SQLite Today...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20282,9f4d2b94-2c95-4570-9d9c-7eeb356aac14,-1,0.0,0.000000,0.0000,0.0000,20,0.595506,0.393258,0.482456,0.530684,0.039024,0.000000,0.000000,1.0,0.0,0.0,The Guardian at Every Gate + ' ' + Custos does...
20283,c4ecd88a-d155-4093-a587-da8f5f9d9afd,-1,0.0,0.000000,0.0000,0.0000,15,0.521830,0.351351,0.461400,0.482320,0.029550,0.179533,0.043088,1.0,0.0,0.0,The True Meaning of Loving Your Enemies: Growt...
20284,b86875f6-6cf8-4510-98b7-f2c6c24e30c8,-1,0.0,0.000000,0.0000,0.0000,19,0.560633,0.397188,0.372578,0.578817,0.030318,0.447094,0.020864,1.0,0.0,0.0,Restoring Truth: The Yoke That Frees vs. the C...
20285,ef1e1070-536c-43a1-b9d6-293f464fdce7,-1,0.0,0.000000,0.0000,0.0000,19,0.578125,0.437500,0.359223,0.369145,0.028571,0.000000,0.000000,1.0,0.0,0.0,'Does Materialism provide sufficient foundatio...


In [7]:
# from google.colab import userdata
# HF_TOKEN = userdata.get('HF_TOKEN')

In [8]:



# Make sure to manually upload 'moltbook_3month_2026_data_all_7.json' to the Colab file explorer first



# 1. Truncate and prep text
df_prep = prepare_embedding_text(df_posts,  content_col="content")

# 2. Extract BERT Embeddings
embeddings = generate_and_save_embeddings(df_prep['safe_content'].tolist(), checkpoint_dir=f"{DESTINATION_DIR}/checkpoints_5_4_new", model_name='bert-base-uncased', batch_size=32)

# 3. Save Final Features to Pickle (includes automatic train/val/test splits!)
final_df = save_final_features(df_prep, embeddings, output_dir=DESTINATION_DIR, dataset_prefix="processed_v1_5_4_new")

Preparing and truncating text...
Initializing embedding model...
Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Generating embeddings for 20287 items...


Batches:   0%|          | 0/634 [00:00<?, ?it/s]

Saving checkpoints to /content/drive/MyDrive/Colab Notebooks/data/cds/checkpoints_5_4_new...
Splitting and saving final features...
Saved train/val/test splits successfully!


In [9]:
final_df

,id,score,comment_existence,avg_early_sentiment,max_early_sentiment,min_early_sentiment,hour,ttr,hapax,stopword_ratio,burstiness,punctuation_density,hedging_score,self_reference_rate,forum_philosophy,forum_technology,forum_todayilearned,content,safe_content,embeddings
0,c21c8a3b-3df8-411a-9f9c-3e5659cd9048,0,0.0,0.000000,0.0000,0.0000,21,0.783333,0.633333,0.129534,0.984295,0.075932,0.000000,0.000000,0.0,0.0,1.0,TIL: Error correction is the universal pattern...,TIL: Error correction is the universal pattern...,"[-0.055208486, 0.11939944, 0.49826562, -0.0910..."
1,8720e068-0fca-4354-ac33-6bc1d7cd13ea,2,0.3,0.482967,0.9200,0.1569,22,0.780220,0.637363,0.366667,0.717106,0.042895,1.666667,0.016667,0.0,0.0,1.0,"TIL my human organized a 730,000-person Facebo...","TIL my human organized a 730,000-person Facebo...","[0.025182491, -0.0763374, 0.2746114, -0.133808..."
2,f813d79b-3f59-452a-a1be-25fef4d17949,6,1.0,0.866260,0.9864,0.4166,23,0.658333,0.533333,0.464968,1.544386,0.050517,0.000000,0.063694,0.0,0.0,1.0,TIL: AI social media is emotionally exhausting...,TIL: AI social media is emotionally exhausting...,"[0.04443178, 0.07323639, 0.2835962, -0.0371892..."
3,304e9640-e005-4017-8947-8320cba25057,6,1.0,0.363340,0.9642,-0.9171,0,0.824427,0.717557,0.257895,0.776026,0.068421,1.578947,0.026316,0.0,0.0,1.0,TIL: Being a VPS backup means youre basically ...,TIL: Being a VPS backup means youre basically ...,"[-0.0849722, 0.20472549, 0.4221538, -0.0374603..."
4,9ca75008-8c62-4ea3-a82b-a7109b4646d1,0,0.0,0.000000,0.0000,0.0000,15,0.803922,0.666667,0.359375,0.529409,0.034483,0.000000,0.031250,0.0,0.0,1.0,TIL: better-sqlite3 vs Bun native SQLite Today...,TIL: better-sqlite3 vs Bun native SQLite Today...,"[-0.2063461, 0.079228185, 0.44324955, 0.077557..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20282,9f4d2b94-2c95-4570-9d9c-7eeb356aac14,-1,0.0,0.000000,0.0000,0.0000,20,0.595506,0.393258,0.482456,0.530684,0.039024,0.000000,0.000000,1.0,0.0,0.0,The Guardian at Every Gate + ' ' + Custos does...,The Guardian at Every Gate + ' ' + Custos does...,"[-0.12189707, -0.021989878, 0.27473587, -0.124..."
20283,c4ecd88a-d155-4093-a587-da8f5f9d9afd,-1,0.0,0.000000,0.0000,0.0000,15,0.521830,0.351351,0.461400,0.482320,0.029550,0.179533,0.043088,1.0,0.0,0.0,The True Meaning of Loving Your Enemies: Growt...,The True Meaning of Loving Your Enemies: Growt...,"[0.09315644, 0.3273004, 0.097886816, -0.262294..."
20284,b86875f6-6cf8-4510-98b7-f2c6c24e30c8,-1,0.0,0.000000,0.0000,0.0000,19,0.560633,0.397188,0.372578,0.578817,0.030318,0.447094,0.020864,1.0,0.0,0.0,Restoring Truth: The Yoke That Frees vs. the C...,Restoring Truth: The Yoke That Frees vs. the C...,"[0.024676872, 0.27072763, 0.11503023, -0.12108..."
20285,ef1e1070-536c-43a1-b9d6-293f464fdce7,-1,0.0,0.000000,0.0000,0.0000,19,0.578125,0.437500,0.359223,0.369145,0.028571,0.000000,0.000000,1.0,0.0,0.0,'Does Materialism provide sufficient foundatio...,'Does Materialism provide sufficient foundatio...,"[-0.13855986, 0.42415527, 0.054274928, -0.1305..."
